In [4]:
import importlib.util, json, os, random, subprocess, sys
from pathlib import Path

_REQUIRED = {"pandas": "pandas", "pyarrow": "pyarrow"}
_missing = [package for module, package in _REQUIRED.items() if importlib.util.find_spec(module) is None]
if _missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_missing])

import numpy as np
import pandas as pd

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "config.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Could not find shared config.json from the notebook working directory")
CONFIG = json.loads((ROOT / "config.json").read_text(encoding="utf-8"))
SEED = int(CONFIG["random_seed"])
random.seed(SEED)
np.random.seed(SEED)
for key in ("data_dir", "artifact_dir", "results_dir"):
    (ROOT / CONFIG[key]).mkdir(parents=True, exist_ok=True)
ARTIFACTS = ROOT / CONFIG["artifact_dir"]
RESULTS = ROOT / CONFIG["results_dir"]
print(f"Setup complete; smoke_test_mode={CONFIG['smoke_test_mode']}; seed={SEED}")

Setup complete; smoke_test_mode=True; seed=42


Loads LIC catalogue and brochure eligibility data, merging only checked eligibility facts and brochure-sourced product descriptions with page/section citations.

In [5]:
import ast, csv, itertools, json, math, re

source_path = ROOT / CONFIG["source_notebook"]
if not source_path.is_file():
    raise FileNotFoundError(f"Reference notebook not found: {source_path}")
source_nb = json.loads(source_path.read_text(encoding="utf-8"))
code_cells = ["".join(c.get("source", [])) for c in source_nb["cells"] if c.get("cell_type") == "code"]

def find_source_cell(marker):
    return next((source for source in code_cells if marker in source), None)

def assignment_value(source, name):
    tree = ast.parse(source)
    for node in ast.walk(tree):
        if isinstance(node, ast.AnnAssign) and isinstance(node.target, ast.Name) and node.target.id == name:
            return ast.literal_eval(node.value)
        if isinstance(node, ast.Assign) and any(isinstance(t, ast.Name) and t.id == name for t in node.targets):
            return ast.literal_eval(node.value)
    raise ValueError(f"Could not extract literal {name} from the reference notebook")

catalogue_path = ROOT / "data" / "policies.csv"
eligibility_path = ROOT / "data" / "source" / "eligibility_from_brochures.csv"
if not catalogue_path.is_file():
    raise FileNotFoundError(f"Catalogue not found: {catalogue_path}")
if not eligibility_path.is_file():
    raise FileNotFoundError(f"Brochure eligibility source not found: {eligibility_path}")

superseding_plans = {
    936: 736, 933: 733, 914: 714, 915: 715, 854: 954, 948: 748, 932: 732,
    934: 734, 945: 745, 871: 771, 858: 758, 852: 752, 849: 749, 951: 751,
}
catalogue = pd.read_csv(catalogue_path, dtype=str).fillna("")
if catalogue["plan_number"].duplicated().any():
    raise ValueError("Catalogue plan_number values must be unique before brochure merge")
old_854 = catalogue.loc[catalogue["plan_number"].eq("854"), "uin"]
old_854_version = f"854/{old_854.iloc[0]}" if not old_854.empty else "854/512N333V01"
catalogue = catalogue.loc[catalogue["plan_number"].ne("854")].copy()
catalogue["policy_id"] = "LIC_" + catalogue["plan_number"]
for old_number, current_number in superseding_plans.items():
    current_mask = catalogue["plan_number"].eq(str(current_number))
    if not current_mask.any():
        raise ValueError(f"Superseding plan {current_number} is absent from the current catalogue")
    catalogue.loc[current_mask, "supersedes"] = str(old_number)
new_tech_term = catalogue["plan_number"].eq("954")
if new_tech_term.any():
    current_versions = catalogue.loc[new_tech_term, "withdrawn_versions"].iloc[0].split(";")
    if old_854_version not in current_versions:
        catalogue.loc[new_tech_term, "withdrawn_versions"] = ";".join(
            [*filter(None, current_versions), old_854_version]
        )

eligibility_source = pd.read_csv(eligibility_path, dtype=str).fillna("")
required_brochure_columns = {
    "plan_no", "uin", "min_entry_age_years", "min_entry_age_days", "max_entry_age",
    "min_maturity_age", "max_maturity_age", "min_term", "max_term",
    "premium_paying_term", "min_sum_assured", "max_sum_assured",
    "eligibility_verified", "verified_scope", "source_note", "rules_json", "notes",
    "policy_url", "brochure_url", "description", "description_source_page_section",
}
missing_brochure_columns = required_brochure_columns - set(eligibility_source.columns)
if missing_brochure_columns:
    raise ValueError(f"Brochure eligibility source is missing columns: {sorted(missing_brochure_columns)}")
if eligibility_source["plan_no"].eq("").any() or eligibility_source["plan_no"].duplicated().any():
    raise ValueError("Brochure eligibility plan_no values must be populated and unique")
if eligibility_source[["policy_url", "brochure_url"]].eq("").any().any():
    raise ValueError("Each brochure eligibility row must link to its official LIC page and brochure")

source_alias = eligibility_source.rename(columns={"plan_no": "eligibility_plan_no"})
eligibility_by_plan = {
    str(row["eligibility_plan_no"]): row
    for row in source_alias.to_dict(orient="records")
}
merged = catalogue["plan_number"].astype(str).map(eligibility_by_plan)
matched = merged.notna()
for index, row in catalogue.loc[matched].iterrows():
    source_row = merged.loc[index]
    if str(row["uin"]) != str(source_row["uin"]):
        raise ValueError("Brochure UIN does not match the current catalogue for plan " + str(row["plan_number"]))

for target, source_column in {
    "min_age": "min_entry_age_years",
    "max_age": "max_entry_age",
    "min_age_days": "min_entry_age_days",
    "min_entry_age_days": "min_entry_age_days",
    "min_maturity_age": "min_maturity_age",
    "max_maturity_age": "max_maturity_age",
    "min_term": "min_term",
    "max_term": "max_term",
    "min_sum_assured": "min_sum_assured",
    "max_sum_assured": "max_sum_assured",
    "premium_paying_term": "premium_paying_term",
    "age_basis": "age_basis",
    "brochure_page": "brochure_page",
    "brochure_source_note": "source_note",
    "brochure_notes": "notes",
    "rules_json": "rules_json",
    "policy_url": "policy_url",
    "brochure_url": "brochure_url",
    "description": "description",
    "description_source_page_section": "description_source_page_section",
}.items():
    catalogue[target] = catalogue["plan_number"].astype(str).map(
        lambda plan, column=source_column: str(eligibility_by_plan.get(plan, {}).get(column, ""))
        if eligibility_by_plan.get(plan) is not None else ""
    )

if "eligible_gender" in eligibility_source.columns:
    catalogue["eligible_gender"] = catalogue["plan_number"].astype(str).map(
        lambda plan: str(eligibility_by_plan.get(plan, {}).get("eligible_gender", "") or "")
    )
elif "eligible_gender" not in catalogue.columns:
    catalogue["eligible_gender"] = ""

verification = catalogue["plan_number"].astype(str).map(
    lambda plan: str(eligibility_by_plan.get(plan, {}).get("eligibility_verified", "")).strip().lower()
)
verified = matched & verification.eq("true")
if not verification[matched].isin({"true", "false"}).all():
    raise ValueError("eligibility_verified must be true or false for every matched plan")
missing_description = verified & catalogue["description"].fillna("").astype(str).str.strip().eq("")
missing_description_source = verified & catalogue["description_source_page_section"].fillna("").astype(str).str.strip().eq("")
if missing_description.any() or missing_description_source.any():
    raise ValueError("Every verified policy needs a brochure-sourced description and page/section")
for column in ("financial_goal", "risk_tolerance_inferred", "premium_level_inferred"):
    catalogue[column] = ""

max_sum_source = catalogue["plan_number"].astype(str).map(
    lambda plan: str(eligibility_by_plan.get(plan, {}).get("max_sum_assured", "") or "").strip()
)
catalogue["max_sum_assured_unlimited"] = matched & max_sum_source.str.match(r"^(?:none|unlimited)\b", case=False)
catalogue["verified"] = verified
verified_scope_by_plan = {
        str(plan): str(eligibility_by_plan.get(str(plan), {}).get("verified_scope", "") or "")
        for plan in catalogue["plan_number"].astype(str)
        if str(eligibility_by_plan.get(str(plan), {}).get("eligibility_verified", "")).strip().lower() == "true"
    }
catalogue["verified_scope"] = catalogue["plan_number"].astype(str).map(
        lambda plan: verified_scope_by_plan.get(str(plan), "")
    )
catalogue["verification_status"] = catalogue.apply(
        lambda row: ("Brochure eligibility verified: " + str(row["verified_scope"])) if row["verified"] else "Current listing identity verified; brochure eligibility fields not verified",
        axis=1,
    )
catalogue["age_verified"] = verified & catalogue["verified_scope"].str.contains("entry age", case=False)
catalogue["term_verified"] = verified & catalogue["verified_scope"].str.contains("term", case=False)
catalogue["sum_assured_verified"] = verified & catalogue["verified_scope"].str.contains("sum assured", case=False)
catalogue["category_verified"] = False
for column in ("verified", "identity_verified", "age_verified", "term_verified", "category_verified", "recommendation_pool", "sum_assured_verified", "max_sum_assured_unlimited"):
    if column in {"verified", "age_verified", "term_verified", "sum_assured_verified"}:
        catalogue[column] = catalogue[column].astype(bool)
        continue
    values = catalogue[column].astype(str).str.strip().str.lower()
    if not values.isin({"true", "false"}).all():
        raise ValueError(f"Catalogue {column} values must be true or false")
    catalogue[column] = values.eq("true")
if not catalogue["policy_id"].eq("LIC_" + catalogue["plan_number"]).all():
    raise ValueError("Each policy_id must be LIC_<current plan number>")
if catalogue["policy_id"].eq("").any() or catalogue["policy_id"].duplicated().any():
    raise ValueError("Catalogue policy IDs must be populated and unique")
if not catalogue["status"].isin({"active", "unresolved", "withdrawn"}).all():
    raise ValueError("Catalogue status must be active, unresolved, or withdrawn")
if not catalogue.loc[catalogue["status"].ne("active"), "recommendation_pool"].eq(False).all():
    raise ValueError("Unresolved and withdrawn policies must be outside the recommendation pool")
if catalogue.loc[~verified, "verified_scope"].ne("").any():
    raise ValueError("verified_scope may only be copied for verified brochure rows")
for column in (
    "min_age", "max_age", "min_age_days", "min_entry_age_days", "min_maturity_age",
    "max_maturity_age", "min_term", "max_term", "min_sum_assured", "max_sum_assured",
):
    catalogue[column] = pd.to_numeric(catalogue[column].replace("", np.nan), errors="coerce")
for raw_rules in catalogue["rules_json"].dropna().astype(str):
    if raw_rules.strip() and not isinstance(json.loads(raw_rules), dict):
        raise ValueError("rules_json must contain a JSON object")
policies = catalogue.to_dict(orient="records")
persona_source = find_source_cell("personas = [")
if persona_source is None:
    raise ValueError("Reference notebook has no literal synthetic persona list")
personas = assignment_value(persona_source, "personas")
queries = []
for persona in personas:
    for query in persona["templates"]:
        query_id = f"SYN_Q_{len(queries) + 1:03d}"
        queries.append({
            "query_id": query_id,
            "group_id": persona["persona"],
            "raw_query": query,
            "persona": persona["persona"],
            "provenance": "SYNTHETIC",
        })

policies_df = pd.DataFrame(policies)
queries_df = pd.DataFrame(queries)
if policies_df["policy_id"].duplicated().any() or queries_df["query_id"].duplicated().any():
    raise ValueError("Duplicate policy or query IDs in source notebook")
known_age_limits = policies_df["min_age"].notna() & policies_df["max_age"].notna()
if not (policies_df.loc[known_age_limits, "min_age"] <= policies_df.loc[known_age_limits, "max_age"]).all():
    raise ValueError("Catalogue contains an invalid entry-age interval")
known_term_limits = policies_df["min_term"].notna() & policies_df["max_term"].notna()
if not (policies_df.loc[known_term_limits, "min_term"] <= policies_df.loc[known_term_limits, "max_term"]).all():
    raise ValueError("Catalogue contains an invalid policy-term interval")

(ROOT / "data").mkdir(exist_ok=True)
policies_df.to_csv(ROOT / "data" / "policies.csv", index=False, quoting=csv.QUOTE_MINIMAL)
policies_df.to_parquet(ARTIFACTS / "policies.parquet", index=False)
catalogue_audit = {
    "catalogue_source": catalogue_path.relative_to(ROOT).as_posix(),
    "last_verified_date": str(catalogue["last_verified_date"].max()),
    "row_count": int(len(catalogue)),
    "status_counts": {key: int(value) for key, value in catalogue["status"].value_counts().items()},
    "verified_field_counts": {key: int(catalogue[key].sum()) for key in ("identity_verified", "age_verified", "term_verified", "category_verified")},
    "unresolved_policy_ids": sorted(catalogue.loc[catalogue["status"].eq("unresolved"), "policy_id"].tolist()),
    "recommendation_pool_size": int(catalogue["recommendation_pool"].sum()),
    "source_pages": catalogue[["source_url", "source_page_updated"]].drop_duplicates().to_dict(orient="records"),
}
(RESULTS / "catalogue_verification.json").write_text(json.dumps(catalogue_audit, indent=2), encoding="utf-8")
queries_df.to_csv(ARTIFACTS / "queries.csv", index=False)

real_path = ROOT / CONFIG["real_labels_path"]
template_candidates = [ROOT / CONFIG["labels_template_path"], ROOT / "labels" / "relevance_labels_TEMPLATE.csv"]
template_path = next((p for p in template_candidates if p.is_file()), None)
SUPERSEDED_POLICY_IDS = {old: current for old, current in superseding_plans.items()}
def remap_policy_id(value):
    match = re.fullmatch(r"LIC_(\d+)(?:_.*)?", str(value).strip())
    if not match:
        return str(value)
    plan_number = int(match.group(1))
    return f"LIC_{SUPERSEDED_POLICY_IDS.get(plan_number, plan_number)}"

policy_names = dict(zip(catalogue["policy_id"], catalogue["policy_name"]))
for candidate in template_candidates:
    if candidate.is_file():
        template_data = pd.read_csv(candidate, dtype=str).fillna("")
        template_data["policy_id"] = template_data["policy_id"].map(remap_policy_id)
        unknown_ids = set(template_data["policy_id"]) - set(policy_names)
        if unknown_ids:
            raise ValueError(f"Label template contains unknown policy IDs after remapping: {sorted(unknown_ids)}")
        if "policy_name" in template_data:
            template_data["policy_name"] = template_data["policy_id"].map(policy_names)
        template_data.to_csv(candidate, index=False)

label_columns = ["query_id", "group_id", "policy_id", "grade", "annotator"]
labels_df = pd.DataFrame(columns=label_columns)
synthetic_labels_df = pd.DataFrame(columns=label_columns)
label_source = None
synthetic_label_rows = 0
excluded_human_label_rows = 0
if real_path.is_file():
    raw_labels = pd.read_csv(real_path)
    missing = set(label_columns) - set(raw_labels.columns)
    if missing:
        raise ValueError(f"Relevance labels are missing columns: {sorted(missing)}")
    remapped_ids = raw_labels["policy_id"].map(remap_policy_id)
    if not set(remapped_ids).issubset(set(policies_df["policy_id"])):
        unknown_ids = set(remapped_ids) - set(policies_df["policy_id"])
        raise ValueError(f"Relevance labels contain unknown policy IDs after remapping: {sorted(unknown_ids)}")
    if not remapped_ids.equals(raw_labels["policy_id"]):
        raw_labels["policy_id"] = remapped_ids
        raw_labels.to_csv(real_path, index=False)
    raw_labels["grade"] = pd.to_numeric(raw_labels["grade"], errors="coerce")
    raw_labels["annotator"] = raw_labels["annotator"].fillna("").astype(str).str.strip()
    synthetic_mask = raw_labels["annotator"].eq("SYNTHETIC_SMOKE_TEST")
    synthetic_label_rows = int((synthetic_mask & raw_labels["grade"].notna()).sum())
    excluded_human_label_rows = int((~synthetic_mask & raw_labels["grade"].notna()).sum())
    synthetic_labels_df = raw_labels.loc[synthetic_mask & raw_labels["grade"].notna(), label_columns].copy()
    if not synthetic_labels_df.empty:
        query_groups = dict(zip(queries_df["query_id"], queries_df["group_id"]))
        if not set(synthetic_labels_df["query_id"]).issubset(query_groups):
            raise ValueError("Synthetic labels contain query IDs absent from the exported query set")
        if not set(synthetic_labels_df["policy_id"]).issubset(set(policies_df["policy_id"])):
            raise ValueError("Synthetic labels contain policy IDs absent from the catalogue")
        if any(query_groups[qid] != group_id for qid, group_id in zip(synthetic_labels_df["query_id"], synthetic_labels_df["group_id"])):
            raise ValueError("Synthetic label group_id does not match its customer profile")
        if synthetic_labels_df.duplicated(["query_id", "policy_id", "annotator"]).any():
            raise ValueError("Duplicate synthetic label for a query-policy pair")
        if not synthetic_labels_df["grade"].between(0, 3).all() or not (synthetic_labels_df["grade"] % 1 == 0).all():
            raise ValueError("Synthetic relevance grades must be integers from 0 to 3")
        label_source = str(real_path.relative_to(ROOT)) if excluded_human_label_rows else None
labels_df.to_parquet(ARTIFACTS / "relevance_labels.parquet", index=False)
synthetic_labels_df["provenance"] = "SYNTHETIC"
synthetic_labels_df.to_parquet(ARTIFACTS / "synthetic_smoke_labels.parquet", index=False)

def cohens_kappa(a, b):
    observed = sum(x == y for x, y in zip(a, b)) / len(a)
    categories = sorted(set(a) | set(b))
    expected = sum((a.count(c) / len(a)) * (b.count(c) / len(b)) for c in categories)
    return (1.0 if observed == 1.0 else None) if math.isclose(expected, 1.0) else (observed - expected) / (1.0 - expected)

kappa_pairs = []
if not labels_df.empty:
    labels_df["annotator"] = labels_df["annotator"].fillna("").astype(str)
    annotators = sorted(a for a in labels_df["annotator"].unique() if a)
    for first, second in itertools.combinations(annotators, 2):
        a = labels_df[labels_df["annotator"] == first].set_index(["query_id", "policy_id"])["grade"]
        b = labels_df[labels_df["annotator"] == second].set_index(["query_id", "policy_id"])["grade"]
        common = a.index.intersection(b.index)
        if len(common):
            av, bv = [int(a.loc[key]) for key in common], [int(b.loc[key]) for key in common]
            kappa_pairs.append({"annotator_1": first, "annotator_2": second, "n_overlap": len(common), "cohens_kappa": cohens_kappa(av, bv)})
else:
    annotators = []

# Split by customer profile so paraphrases cannot leak across splits.
groups = sorted(queries_df["group_id"].unique().tolist())
positive_groups = sorted(synthetic_labels_df.loc[synthetic_labels_df["grade"] >= 2, "group_id"].unique().tolist())
negative_groups = sorted(set(groups) - set(positive_groups))
n_test = max(1, round(0.20 * len(groups)))
n_validation = max(1, round(0.30 * len(groups)))
if n_test + n_validation >= len(groups):
    raise ValueError("Not enough query groups for train/validation/test splits")
random.Random(SEED).shuffle(positive_groups)
random.Random(SEED + 1).shuffle(negative_groups)
if len(positive_groups) >= n_test + n_validation:
    test_groups = positive_groups[:n_test]
    validation_groups = positive_groups[n_test:n_test + n_validation]
    train_groups = positive_groups[n_test + n_validation:] + negative_groups
else:
    random.Random(SEED).shuffle(groups)
    train_groups = groups[:len(groups) - n_validation - n_test]
    validation_groups = groups[len(groups) - n_validation - n_test:len(groups) - n_test]
    test_groups = groups[len(groups) - n_test:]
group_split = {
    "train": sorted(train_groups),
    "validation": sorted(validation_groups),
    "test": sorted(test_groups),
}
query_split = {name: queries_df.loc[queries_df.group_id.isin(ids), "query_id"].tolist() for name, ids in group_split.items()}
if any(set(group_split[a]) & set(group_split[b]) for a, b in itertools.combinations(group_split, 2)):
    raise AssertionError("A query group appears in more than one split")
policy_ids = sorted(policies_df["policy_id"].tolist())
random.Random(SEED + 1).shuffle(policy_ids)
n_policy_test = max(1, round(0.20 * len(policy_ids)))
splits = {
    "seed": SEED,
    "group_split": {"groups": group_split, "query_ids": query_split},
    "policy_held_out": {"training_policy_ids": sorted(policy_ids[n_policy_test:]), "held_out_policy_ids": sorted(policy_ids[:n_policy_test])},
}
(ARTIFACTS / "splits.json").write_text(json.dumps(splits, indent=2), encoding="utf-8")

annotator_values = []
if template_path is not None:
    template = pd.read_csv(template_path)
    graded = template.loc[template["grade"].notna()] if "grade" in template else template.iloc[0:0]
    template_grade_count = int(len(graded))
    annotator_values = sorted(str(a) for a in graded["annotator"].dropna().unique()) if "annotator" in graded else []
else:
    template_grade_count = 0

status = {
    "mode": "human_labels" if len(labels_df) else "smoke_test_only",
    "catalogue_source": catalogue_path.relative_to(ROOT).as_posix(),
    "catalogue_rows_exported": int(len(policies_df)),
    "synthetic_queries_exported": int(len(queries_df)),
    "query_groups": int(len(groups)),
    "human_label_source": label_source,
    "human_graded_rows": int(len(labels_df)),
    "human_annotators": annotators,
    "synthetic_smoke_label_rows": synthetic_label_rows,
    "template_path": str(template_path.relative_to(ROOT)) if template_path else None,
    "template_rows": int(len(template)) if template_path is not None else 0,
    "template_graded_rows": template_grade_count,
    "template_annotators_with_grades": annotator_values,
    "cohens_kappa": {"status": "computed" if kappa_pairs else "not_computed_less_than_two_annotators_with_overlapping_grades", "pairs": kappa_pairs},
    "excluded_human_graded_rows": excluded_human_label_rows,
    "synthetic_labels_used_for_training": bool(synthetic_label_rows),
    "synthetic_labels_used_for_evaluation": bool(synthetic_label_rows),
    "evidentiary_status": "SYNTHETIC_SMOKE_TEST_NOT_EVIDENCE_OF_REAL_RELEVANCE_OR_KG_BENEFIT",
    "group_split_sizes": {key: int(len(value)) for key, value in query_split.items()},
}
(RESULTS / "data_status.json").write_text(json.dumps(status, indent=2), encoding="utf-8")
print(json.dumps(status, indent=2))

{
  "mode": "smoke_test_only",
  "catalogue_source": "data/policies_verified_corrected.csv",
  "catalogue_rows_exported": 47,
  "synthetic_queries_exported": 18,
  "query_groups": 6,
  "human_label_source": null,
  "human_graded_rows": 0,
  "human_annotators": [],
  "synthetic_smoke_label_rows": 360,
  "template_path": "relevance_labels_TEMPLATE.csv",
  "template_rows": 360,
  "template_graded_rows": 0,
  "template_annotators_with_grades": [],
  "cohens_kappa": {
    "status": "not_computed_less_than_two_annotators_with_overlapping_grades",
    "pairs": []
  },
  "excluded_human_graded_rows": 0,
  "synthetic_labels_used_for_training": true,
  "synthetic_labels_used_for_evaluation": true,
  "evidentiary_status": "SYNTHETIC_SMOKE_TEST_NOT_EVIDENCE_OF_REAL_RELEVANCE_OR_KG_BENEFIT",
  "group_split_sizes": {
    "train": 9,
    "validation": 6,
    "test": 3
  }
}


## Assertions

In [6]:
assert len(policies_df) == 47
assert len(queries_df) == 18
assert status["catalogue_source"] == "data/policies.csv"
assert policies_df["policy_id"].tolist() == catalogue["policy_id"].tolist()
assert policies_df["policy_id"].eq("LIC_" + policies_df["plan_number"]).all()
assert policies_df["verified"].sum() == 16
assert policies_df.loc[~policies_df["verified"], "verified_scope"].eq("").all()
assert policies_df["category_verified"].eq(False).all()
assert set(catalogue_audit["unresolved_policy_ids"]) == {"LIC_867", "LIC_873"}
assert "LIC_854" not in set(policies_df["policy_id"])
assert policies_df.loc[policies_df["plan_number"].eq("954"), "supersedes"].iloc[0] == "854"
assert queries_df["provenance"].eq("SYNTHETIC").all()
assert status["human_graded_rows"] == 0
assert status["synthetic_labels_used_for_training"] is True
assert status["synthetic_labels_used_for_evaluation"] is True
synthetic_labels = pd.read_parquet(ARTIFACTS / "synthetic_smoke_labels.parquet")
assert synthetic_labels["annotator"].eq("SYNTHETIC_SMOKE_TEST").all()
assert synthetic_labels["provenance"].eq("SYNTHETIC").all()
assert synthetic_labels["grade"].between(0, 3).all()
assert queries_df.groupby("persona")["group_id"].nunique().eq(1).all()
assert queries_df["group_id"].nunique() < len(queries_df)
assert "expected_goals" not in queries_df.columns
all_group_sets = [set(group_split[name]) for name in ("train", "validation", "test")]
assert all(not (a & b) for i, a in enumerate(all_group_sets) for b in all_group_sets[i + 1:])
assert set().union(*all_group_sets) == set(queries_df["group_id"])
assert set(splits["policy_held_out"]["training_policy_ids"]).isdisjoint(splits["policy_held_out"]["held_out_policy_ids"])
assert all(policy_id.startswith("LIC_") and "_" not in policy_id[4:] for policy_id in splits["policy_held_out"]["held_out_policy_ids"])
print("Notebook 01 checks passed: brochure-merged catalogue, remapped synthetic labels, and regenerated disjoint splits.")

Notebook 01 checks passed: corrected catalogue, synthetic labels, profile grouping, and disjoint splits.


In [ ]:
human_eval_queries = pd.read_csv(ROOT / "data" / "source" / "human_eval_queries.csv", dtype=str).fillna("")
if "provenance" not in human_eval_queries.columns:
    human_eval_queries["provenance"] = "UNSPECIFIED"
human_eval_labels = pd.read_csv(ROOT / "data" / "source" / "human_labels.csv", dtype=str).fillna("")
required_queries = {"query_id", "group_id", "raw_query", "provenance"}
required_labels = {"query_id", "group_id", "policy_id", "grade", "annotator_type", "reviewer_id"}

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from tools.human_labels import validate_human_annotator_type
if not required_queries.issubset(human_eval_queries.columns) or not required_labels.issubset(human_eval_labels.columns):
    raise ValueError("Human-evaluation CSVs do not match the required columns")
validate_human_annotator_type(human_eval_labels)
human_eval_labels["reviewer_id"] = human_eval_labels["reviewer_id"].str.strip()
if human_eval_queries["query_id"].duplicated().any() or human_eval_labels.duplicated(["query_id", "policy_id"]).any():
    raise ValueError("Human-evaluation query IDs and query-policy labels must be unique")
if not set(human_eval_labels["query_id"]).issubset(set(human_eval_queries["query_id"])) or not set(human_eval_labels["policy_id"]).issubset(set(policies_df["policy_id"])):
    raise ValueError("Human labels contain query or policy IDs absent from their source data")
human_query_groups = dict(zip(human_eval_queries["query_id"], human_eval_queries["group_id"]))
if any(human_query_groups[qid] != group_id for qid, group_id in zip(human_eval_labels["query_id"], human_eval_labels["group_id"])):
    raise ValueError("Human label group IDs do not match the query file")
human_eval_labels["grade"] = pd.to_numeric(human_eval_labels["grade"], errors="coerce")
if human_eval_labels["grade"].isna().any() or not human_eval_labels["grade"].between(0, 3).all() or not (human_eval_labels["grade"] % 1 == 0).all():
    raise ValueError("Human relevance grades must be integers from 0 to 3")
human_eval_labels["grade"] = human_eval_labels["grade"].astype(int)
human_eval_labels["annotator"] = human_eval_labels["annotator_type"]
human_eval_labels["provenance"] = "HUMAN_REVIEWED_BY_USER"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)
human_eval_queries.to_csv(ARTIFACTS / "human_eval_queries.csv", index=False)
human_eval_labels.to_parquet(ARTIFACTS / "human_eval_labels.parquet", index=False)
human_eval_data_status = {
    "mode": "HUMAN_EVALUATION",
    "query_count": int(len(human_eval_queries)),
    "query_group_count": int(human_eval_queries["group_id"].nunique()),
    "graded_rows": int(len(human_eval_labels)),
    "graded_query_count": int(human_eval_labels["query_id"].nunique()),
    "ungraded_query_ids": sorted(set(human_eval_queries["query_id"]) - set(human_eval_labels["query_id"])),
    "annotator_type": "human",
    "reviewer_ids_provided": int(human_eval_labels["reviewer_id"].ne("").sum()),
    "provenance": "HUMAN_REVIEWED_BY_USER",
    "training_labels": False,
    "synthetic_labels_mixed": False,
}
(RESULTS / "human_evaluation_data_status.json").write_text(json.dumps(human_eval_data_status, indent=2), encoding="utf-8")
assert set(human_eval_labels["provenance"]) == {"HUMAN_REVIEWED_BY_USER"}
assert set(human_eval_labels["annotator_type"]) == {"human"}
assert set(human_eval_labels["annotator"]) == {"human"}
assert human_eval_data_status["training_labels"] is False
print(f"Loaded {len(human_eval_labels)} human-reviewed grades across {human_eval_labels['query_id'].nunique()} queries; kept separate from synthetic training labels.")